# CHIRPS v3 monthly precipitation

Inspect final monthly CHIRPS v3 precipitation for January 2020 near Nairobi. CHIRPS combines satellite-derived estimates and station observations into gridded monthly rainfall totals at approximately 0.05° resolution. Values represent precipitation over grid cells, in millimetres, not point observations or design-rainfall intensities. The gridded source is not a flood-depth or impact dataset. No daily disaggregation is used here.

The companion [pipeline](../pipelines/chirps_monthly_pipeline.py) fetches and checksums the full monthly COG once, then reads only the requested pixel window. The source's `-9999` ocean/no-data sentinel is omitted; valid zero rainfall is retained. The requested bounds select a raster window and the coordinates are pixel centres. The result is not an area-weighted average. The snapshot cache and manifest permit offline replay; a source version label alone does not fix bytes.

Source: [Climate Hazards Center CHIRPS3 Data Repository](https://doi.org/10.15780/G2JQ0P), CC BY 4.0; [dataset description](https://www.chc.ucsb.edu/data/chirps3).

Run from a local crc-docs checkout: use `uv sync`, then launch Jupyter with
`uv run jupyter lab` and select the project kernel. The first pipeline run
requires internet access.


In [1]:
# Check the local notebook environment before running the pipeline.
import importlib.util
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (root / "pipelines").is_dir():
    raise RuntimeError("Run this notebook from the crc-docs checkout or its notebooks directory.")
missing = [name for name in ("crc_sdk", "pyarrow", "pandas")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Run `uv sync` from the crc-docs checkout and select its kernel: {missing}")
sys.path.insert(0, str(root))
from pipelines.chirps_monthly_pipeline import run
import pyarrow.parquet as pq

output = root / "pipeline_output" / "chirps_monthly"
result = run(output)
print("Valid grid cells:", result["valid_pixels"])
print("Snapshot SHA-256:", result["source"]["sha256"])
print("Retrieved:", result["source"]["retrieved_at"])

Valid grid cells: 4
Snapshot SHA-256: b0c7a1520e99d355253645a5f97688a9598f37ccba9370a4d4c9753df88d2458
Retrieved: 2026-10-30T19:10:52.043790+00:00


In [2]:
pixels = pq.read_table(output / "monthly.parquet").to_pandas()
pixels.sort_values(["latitude", "longitude"])

,longitude,latitude,precipitation_mm,month
2,36.775003,-1.325001,181.057510,2020-01-01
3,36.825003,-1.325001,172.634293,2020-01-01
0,36.775003,-1.275001,166.290787,2020-01-01
1,36.825003,-1.275001,148.933212,2020-01-01


Pixel values vary across the selected window. A single month cannot establish an anomaly or long-term trend; those require an explicitly chosen comparable baseline, completeness rules and a spatial aggregation policy. Daily CHIRPS v3 products have separate reanalysis and satellite disaggregation methods and should not be interpreted as directly observed subdaily rainfall.